In [1]:
!pip install -U langchain
!pip install -U langchain-cohere
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-chroma
!pip install -U pypdf
!pip install -U chromadb

  Using cached langchain_cohere-0.6.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached cohere-5.21.1-py3-none-any.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 9.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 53.7 MB/s eta 0:00:00a 0:00:01
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_classic-1.0.8-py3-none-any.whl (1.0 MB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
  Attempting uninstall: requests
    Found existing installation:

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
key = user_secrets.get_secret("coherekey")

----

## **Part 3 — Multi-PDF RAG**

In [38]:
from langchain_community.document_loaders import PyPDFLoader
import os

# Define the list of PDF files you want to load
pdf_files = ["/kaggle/input/datasets/islamohamed10/data-structure/Part 2 Linked List.pdf", "/kaggle/input/datasets/islamohamed10/data-structure/Part 3  Stack.pdf","/kaggle/input/datasets/islamohamed10/data-structure/Part 4 Queue.pdf","/kaggle/input/datasets/islamohamed10/data-structure/Part 5 Recursion.pdf"]

all_documents = []

for pdf_path in pdf_files:
    if os.path.exists(pdf_path):
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        all_documents.extend(docs)
        print(f"Loaded {pdf_path}: {len(docs)} pages")
    else:
        print(f"Warning: File not found at {pdf_path}")

print(f"\nTotal documents loaded: {len(all_documents)}")

Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 2 Linked List.pdf: 29 pages
Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 3  Stack.pdf: 18 pages
Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 4 Queue.pdf: 27 pages
Loaded /kaggle/input/datasets/islamohamed10/data-structure/Part 5 Recursion.pdf: 14 pages

Total documents loaded: 88


In [39]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)


In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split all loaded documents into chunks
chunks = text_splitter.split_documents(all_documents)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 113


In [41]:
from langchain_chroma import Chroma
import time

# To avoid the 429 Too Many Requests error with Trial keys,
# we decrease batch size and increase sleep time to stay under token limits.
batch_size = 30
vectorstore = Chroma(collection_name="pdf_rag", embedding_function=embeddings)

for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    vectorstore.add_documents(batch)
    print(f"Processed {i + len(batch)} / {len(chunks)} chunks...")
    if i + batch_size < len(chunks):
        # Trial keys often require a significant pause to reset token-per-minute counters
        time.sleep(3)

Processed 30 / 113 chunks...
Processed 60 / 113 chunks...
Processed 90 / 113 chunks...
Processed 113 / 113 chunks...


In [42]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [43]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
xxvi   Contents in Detail
IDLE  .  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on Linux . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on OS X  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 496
Installing IDLE on Windows  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
Customizing IDLE Settings  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
Emacs and vim  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 497
C 
gettIng helP 499
First Steps  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 499
Try It Again  . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

In [44]:
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


In [45]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

In [46]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [47]:
question = "what is the linkedlist?"

response = rag_chain.invoke(question)

print(response.content)

Based on the provided context, a **LinkedList** is a data structure composed of a sequence of elements called **links** or **nodes**. Each link contains data and a reference (or pointer) to the next link in the sequence. The **LinkList** class, for example, contains a reference to the first link (`first`) in the list. 

Here are the key characteristics of a LinkedList from the context:

1. **Link Class**: Each link in the list contains data (e.g., `iData`, `dData`, or `dData`) and a reference to the next link (`next`).
2. **LinkList Class**: Manages the list by holding a reference to the first link (`first`). It includes methods like `isEmpty()` to check if the list is empty.
3. **FirstLastList Class**: An extension of the basic LinkedList that also maintains a reference to the last link (`last`), allowing for efficient insertion and deletion at both ends of the list.

In summary, a LinkedList is a linear collection of nodes where each node points to the next node in the sequence.


In [48]:
question = "what is the stack?"

response = rag_chain.invoke(question)

print(response.content)

A stack is a LIFO (Last In, First Out) data structure that allows access to only one data item: the last item inserted. If you remove this item, you can access the next-to-last item inserted, and so on.


In [49]:
question = "who is mohamed salah?"

response = rag_chain.invoke(question)

print(response.content)

I don't know. The context provided does not contain any information about Mohamed Salah. It appears to be a resume or profile for a person named Islam Mohamed, who is a Computer and Communication Engineering student with a focus on Artificial Intelligence and Machine Learning. The rest of the context includes Python code examples and explanations related to programming concepts such as lists, slicing, and loops.
